<a href="https://colab.research.google.com/github/derekhatherley/A03-knn/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting?
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

1. Regression predicts a numeric outcome, while a classification predicts a category.  
2. A confusion table cross counts predictions by its actual class and its predicted class. It allows us to understand if a model is accurately predicting classes (True positive/negative), or is producing false positives or false negatives. It allows us to see how often a model makes a mistake and what kind of mistake it is making.
3. SSE sums the squared errors for many valid observations. A low value means the there are smaller errors on that validatiomn set, while a high value means a higher error.
4. Overfitting is when a model fits the training data too closely, and fails to extrapolate those trends well to other data. Underfitting is when a model is too simple to recognize important trends in the data.
5. Evaluating different values of k on the accuracy on the test set determines an optimal value to generalize to new observations beyond just the training set. In that case, the model doesn't just reflect the training set and can be used for new observations in a test set.
6. A class label as a prediction gives just one label as its response, while a probability distribution gives the probability of each label. The weakness of the class label prediction is that it does not convey uncertainty in its prediction, but its advantage is that its prediction is easy to interpret. The weakness of the probability distribution is that they are difficult to interpret and does not give you a discrete answer, while the strength is that it represents all possibilities based on the training set.  

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Derek Hatherley

**Date:** 10/1/26

In [47]:
# Your Phase 1 solution to the problem goes here.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, accuracy_score

df = pd.read_csv('land_mines.csv')

# 1 EDA

print('Data frame length:', len(df)) # 338 Data points
mine_types = df['mine_type'].value_counts().sort_index()
print(mine_types) # Pretty even distribution of mine types
print(df.isna().sum()) # No missing values
print(df.groupby('mine_type')[['voltage', 'height', 'soil']].mean())
# Averages suggest that voltage varies most across mine types

# 2 Split and Scale

eval_x = df[['voltage', 'height', 'soil']]
eval_y = df['mine_type']
x_train, x_valid, y_train, y_valid = train_test_split(eval_x, eval_y, test_size=0.5, random_state=65)
# set random state to get same result every time

eval_scaler = MinMaxScaler()
eval_X_train_scaled = eval_scaler.fit_transform(x_train)
eval_X_valid_scaled = eval_scaler.transform(x_valid)

#3 kkn

eval_ks = np.arange(1, 51)
eval_valid_sse = []
eval_train_sse = []
for eval_k in eval_ks:
    eval_candidate = KNeighborsRegressor(n_neighbors=int(eval_k))
    eval_candidate.fit(eval_X_train_scaled, y_train)
    eval_valid_hat = eval_candidate.predict(eval_X_valid_scaled)
    eval_train_hat = eval_candidate.predict(eval_X_train_scaled)
    eval_valid_sse.append(np.sum(
        (y_valid.to_numpy() - eval_valid_hat) ** 2
    ))
    eval_train_sse.append(np.sum(
        (y_train.to_numpy() - eval_train_hat) ** 2
    ))
eval_k_star = int(eval_ks[np.argmin(eval_valid_sse)])
# Find optimal k value as demonstrated in lecture slides
print()
print('Optimal k:', eval_k_star) # Optimal k

# 4 Confusion Table

knn = KNeighborsClassifier(n_neighbors = eval_k_star).fit(eval_X_train_scaled, y_train)
prediction = knn.predict(eval_X_valid_scaled)

labels = sorted(eval_y.unique())
confusion = pd.DataFrame(confusion_matrix(y_valid, prediction, labels = labels),
                         index=[f'Actual {l}' for l in labels],
                         columns=[f'Pred {l}' for l in labels])
print(confusion)
accuracy = accuracy_score(y_valid, prediction)
print('Accuracy:', round(accuracy,3)*100, '%')
# Overall accuracy is 44.4%
acc_by_type = pd.DataFrame({
    'Accuracy by Type (%)': 100*(np.diag(confusion) / confusion.sum(axis=0).values).round(3),
}, index = labels)
print()
print(acc_by_type)
# Low accuracy for all types except for 2

# 5 Interpretation

# Even though this model was made with the optimal number of known nearest neighbors, it only has
# substantial accuracy in predicting type 2 mines. In practice, this model could be used by some agency,
# but its output should not be considered with any confidence unless it predicts a type 2 mine. In cases
# where the model does not predict type 2, it may be useful to add a regression in addition to the
# classification. Since the model cannot reliably predict the mine type, showing a probability is far
# more useful, and does not relay a false confidence in the prediction.

Data frame length: 338
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64
voltage      0
height       0
soil         0
mine_type    0
dtype: int64
            voltage    height      soil
mine_type                              
1          0.296463  0.495519  0.492958
2          0.721123  0.487013  0.508571
3          0.402408  0.527548  0.496970
4          0.345365  0.506887  0.503030
5          0.379598  0.530070  0.516923

Optimal k: 11
          Pred 1  Pred 2  Pred 3  Pred 4  Pred 5
Actual 1      26       0       3       2       3
Actual 2       0      30       6       0       2
Actual 3       3       2      10       2      13
Actual 4      14       0      13       4       7
Actual 5      11       0      11       2       5
Accuracy: 44.4 %

   Accuracy by Type (%)
1                  48.1
2                  93.8
3                  23.3
4                  40.0
5                  16.7


## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** [To be provided]

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [46]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]